# Kalkulasi Manual: Pipeline Prediksi Harga Perak
## Prophet + XGBoost / LightGBM — Step-by-Step Walkthrough

Notebook ini menampilkan **seluruh perhitungan** pipeline prediksi harga perak secara transparan
menggunakan 10 data riil dari Yahoo Finance.

Pipeline mengikuti implementasi `forecaster.py` (versi web app).

**Struktur:**
1. Download & seleksi data
2. Feature engineering (lag, rolling, RSI, MACD, Bollinger, ATR, cross-asset, kalender)
3. Prophet fitting dan dekomposisi komponen
4. Log return target
5. Training XGBoost & LightGBM
6. Prediksi dan konversi ke harga
7. Evaluasi metrik (RMSE, MAE, MAPE, R2)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
import math
from prophet import Prophet
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)
print("Semua library berhasil diimpor.")

Importing plotly failed. Interactive plots will not work.


Semua library berhasil diimpor.


## 1. Download Data dari Yahoo Finance

Download 6 bulan data historis (~130 hari trading) untuk 4 variabel:

| Ticker | Variabel | Satuan |
|--------|----------|--------|
| `SI=F` | Harga Perak | USD/troy oz |
| `GC=F` | Harga Emas | USD/troy oz |
| `CL=F` | Harga Minyak WTI | USD/barrel |
| `DX-Y.NYB` | Indeks Dolar AS | USD Index |

Data 6 bulan diperlukan agar fitur lag_14 dan rolling_30 dapat dihitung dengan benar.

In [2]:
TICKERS_MAP = {'SI=F': 'silver', 'GC=F': 'gold', 'CL=F': 'oil', 'DX-Y.NYB': 'usd'}
SERIES_LIST = ['silver', 'gold', 'oil', 'usd']
LAG_DAYS        = [1, 2, 3, 5, 7, 10, 14]
ROLLING_WINDOWS = [5, 10, 20, 30]

print('Mengunduh data dari Yahoo Finance...')
raw = yf.download(
    list(TICKERS_MAP.keys()),
    period='6mo',
    interval='1d',
    auto_adjust=True,
    progress=False
)

close_raw = raw['Close'].copy()
close_raw = close_raw.rename(columns=TICKERS_MAP)[SERIES_LIST]
close_raw = close_raw.dropna().reset_index()
close_raw = close_raw.rename(columns={'Date': 'ds'})
close_raw['ds'] = pd.to_datetime(close_raw['ds'])
if close_raw['ds'].dt.tz is not None:
    close_raw['ds'] = close_raw['ds'].dt.tz_localize(None)

print(f'Total baris berhasil diunduh: {len(close_raw)}')
print(f'Rentang tanggal: {close_raw["ds"].min().date()} s/d {close_raw["ds"].max().date()}')
print()
print('15 baris terakhir:')
close_raw[['ds','silver','gold','oil','usd']].tail(15).reset_index(drop=True)

Mengunduh data dari Yahoo Finance...
Total baris berhasil diunduh: 124
Rentang tanggal: 2025-11-07 s/d 2026-05-07

15 baris terakhir:


Ticker,ds,silver,gold,oil,usd
0,2026-04-17,81.737999,4857.600098,83.849998,98.099998
1,2026-04-20,79.950996,4806.600098,89.610001,98.050003
2,2026-04-21,76.411003,4698.399902,92.129997,98.410004
3,2026-04-22,77.892998,4732.500000,92.959999,98.589996
4,2026-04-23,75.464996,4705.100098,95.849998,98.800003
5,2026-04-24,76.383003,4722.299805,94.400002,98.510002
6,2026-04-27,75.002998,4675.399902,96.370003,98.480003
7,2026-04-28,73.205002,4591.500000,99.930000,98.620003
8,2026-04-29,71.569000,4545.200195,106.879997,98.919998
9,2026-04-30,73.533997,4614.700195,105.070000,98.080002


## 2. Seleksi Sampel Demo (10 Baris Terakhir)

Fitur dihitung dari seluruh dataset agar lag dan rolling terkompilasi dengan benar.
Kemudian kita ambil **10 baris terakhir** sebagai sampel demonstrasi.

**Split:**
- Baris 0–7 (8 baris) → **Training set**
- Baris 8–9 (2 baris) → **Test set** (evaluasi)

In [3]:
df_all = close_raw.copy().reset_index(drop=True)
N_SAMPLE = 10
N_TRAIN  = 8
N_TEST   = 2

print(f'Dataset total: {len(df_all)} baris')
print(f'Demo: 10 baris terakhir  |  Train: {N_TRAIN}  |  Test: {N_TEST}')
print()

# Compute all non-Prophet features on full dataset
df = df_all.copy()

feat_dict = {}
for series in SERIES_LIST:
    ret = df[series].pct_change()
    for lag in LAG_DAYS:
        feat_dict[f'{series}_lag_{lag}']     = df[series].shift(lag)
        feat_dict[f'{series}_ret_lag_{lag}'] = ret.shift(lag)
    for w in ROLLING_WINDOWS:
        feat_dict[f'{series}_roll_mean_{w}'] = df[series].rolling(w).mean()
        feat_dict[f'{series}_roll_std_{w}']  = df[series].rolling(w).std()

# Technical indicators (all shifted by 1 to avoid look-ahead)
delta    = df['silver'].diff()
up, down = delta.clip(lower=0), (-delta).clip(lower=0)
avg_up14 = up.rolling(14).mean()
avg_dn14 = down.rolling(14).mean().replace(0, 1e-9)
feat_dict['silver_rsi_14'] = (100 - 100 / (1 + avg_up14 / avg_dn14)).shift(1)

ema12 = df['silver'].ewm(span=12, adjust=False).mean()
ema26 = df['silver'].ewm(span=26, adjust=False).mean()
macd  = ema12 - ema26
feat_dict['silver_macd']        = macd.shift(1)
feat_dict['silver_macd_signal'] = macd.ewm(span=9, adjust=False).mean().shift(1)

rm20 = df['silver'].rolling(20).mean()
rs20 = df['silver'].rolling(20).std().replace(0, 1e-9)
feat_dict['silver_bb_pos'] = ((df['silver'] - rm20) / rs20).shift(1)
feat_dict['silver_atr_14'] = df['silver'].diff().abs().rolling(14).mean().shift(1)

gs = df['gold']   / df['silver'].replace(0, 1e-9)
so = df['silver'] / df['oil'].replace(0, 1e-9)
feat_dict['gold_silver_ratio_lag_1'] = gs.shift(1)
feat_dict['gold_silver_ratio_lag_5'] = gs.shift(5)
feat_dict['silver_oil_ratio_lag_1']  = so.shift(1)

feat_dict['day_of_week'] = df['ds'].dt.dayofweek
feat_dict['month']       = df['ds'].dt.month

df_feats = pd.concat([df, pd.DataFrame(feat_dict, index=df.index)], axis=1)
df_clean = df_feats.dropna().reset_index(drop=True)
df_demo  = df_clean.tail(N_SAMPLE).reset_index(drop=True)

print(f'Baris setelah dropna: {len(df_clean)}')
print(f'Sampel demo ({N_SAMPLE} baris):')
print(df_demo[['ds','silver','gold','oil','usd']].to_string())

Dataset total: 124 baris
Demo: 10 baris terakhir  |  Train: 8  |  Test: 2

Baris setelah dropna: 95
Sampel demo (10 baris):
          ds    silver        gold        oil       usd
0 2026-04-24 76.383003 4722.299805  94.400002 98.510002
1 2026-04-27 75.002998 4675.399902  96.370003 98.480003
2 2026-04-28 73.205002 4591.500000  99.930000 98.620003
3 2026-04-29 71.569000 4545.200195 106.879997 98.919998
4 2026-04-30 73.533997 4614.700195 105.070000 98.080002
5 2026-05-01 75.950996 4629.899902 101.940002 98.209999
6 2026-05-04 73.071999 4519.500000 106.419998 98.470001
7 2026-05-05 73.108002 4555.799805 102.269997 98.480003
8 2026-05-06 76.810997 4681.899902  95.080002 98.019997
9 2026-05-07 80.970001 4747.299805  92.889999 97.875000


## 3. Feature Engineering: Lag Features

Untuk setiap 4 variabel, kita buat:
- **Price lag**: `{series}_lag_{d}[t] = price[t - d]`
- **Return lag**: `{series}_ret_lag_{d}[t] = (price[t-d] - price[t-d-1]) / price[t-d-1]`

Lag days: d ∈ {1, 2, 3, 5, 7, 10, 14}

Total: 4 variabel × 7 lag × 2 (price + return) = **56 fitur**

In [4]:
# Silver price lag features (demo rows only)
lag_cols_price = [f'silver_lag_{d}' for d in LAG_DAYS]
lag_cols_ret   = [f'silver_ret_lag_{d}' for d in LAG_DAYS]

lag_df = df_demo[['ds','silver'] + lag_cols_price + lag_cols_ret].copy()
print('Silver Lag Features (10 baris demo):')
print(lag_df.to_string())
print()

# Manual verification for last row
last_full_idx = len(df_clean) - 1
print('Verifikasi baris terakhir (indeks 9 dalam demo):')
sv = df_clean['silver']
for d in [1, 2, 3]:
    price_d = df_clean['silver'].iloc[last_full_idx - d]
    price_d1 = df_clean['silver'].iloc[last_full_idx - d - 1]
    ret_d = (price_d - price_d1) / price_d1
    print(f'  silver_lag_{d}     = silver[t-{d}] = {price_d:.6f}')
    print(f'  silver_ret_lag_{d} = ({price_d:.4f} - {price_d1:.4f}) / {price_d1:.4f} = {ret_d:.8f}')

Silver Lag Features (10 baris demo):
          ds    silver  silver_lag_1  silver_lag_2  silver_lag_3  silver_lag_5  silver_lag_7  silver_lag_10  silver_lag_14  silver_ret_lag_1  silver_ret_lag_2  silver_ret_lag_3  silver_ret_lag_5  silver_ret_lag_7  silver_ret_lag_10  silver_ret_lag_14
0 2026-04-24 76.383003     75.464996     77.892998     76.411003     81.737999     79.490997      76.323997      72.661003         -0.031171          0.019395         -0.044277          0.039844          0.001260           0.000616          -0.001017
1 2026-04-27 75.002998     76.383003     75.464996     77.892998     79.950996     78.606003      75.523003      71.825996          0.012165         -0.031171          0.019395         -0.021863         -0.011133          -0.010495          -0.011492
2 2026-04-28 73.205002     75.002998     76.383003     75.464996     76.411003     81.737999      79.390999      75.223999         -0.018067          0.012165         -0.031171         -0.044277          0.0398

## 4. Feature Engineering: Rolling Statistics

Untuk setiap variabel:
- `roll_mean_w[t] = mean(price[t-w+1 .. t])`
- `roll_std_w[t]  = std(price[t-w+1 .. t])`

Window w ∈ {5, 10, 20, 30}

Total: 4 × 4 × 2 = **32 fitur**

In [5]:
roll_cols_show = ([f'silver_roll_mean_{w}' for w in ROLLING_WINDOWS]
                + [f'silver_roll_std_{w}'  for w in ROLLING_WINDOWS])

roll_df = df_demo[['ds','silver'] + roll_cols_show].copy()
print('Silver Rolling Features (10 baris demo):')
print(roll_df.to_string())
print()

# Manual verification: roll_mean_5 for last row
last_idx = len(df_clean) - 1
w5_data = df_clean['silver'].iloc[last_idx-4:last_idx+1].values
print(f'Verifikasi silver_roll_mean_5 baris terakhir:')
print(f'  5 harga silver: {[round(x,4) for x in w5_data]}')
print(f'  mean = sum / 5 = {sum(w5_data):.6f} / 5 = {np.mean(w5_data):.6f}')
print()
w30_data = df_clean['silver'].iloc[last_idx-29:last_idx+1].values
print(f'Verifikasi silver_roll_std_30 baris terakhir (ddof=1):')
print(f'  mean_30 = {np.mean(w30_data):.6f}')
print(f'  std_30  = {np.std(w30_data, ddof=1):.6f}')

Silver Rolling Features (10 baris demo):
          ds    silver  silver_roll_mean_5  silver_roll_mean_10  silver_roll_mean_20  silver_roll_mean_30  silver_roll_std_5  silver_roll_std_10  silver_roll_std_20  silver_roll_std_30
0 2026-04-24 76.383003           77.220599            78.085200            75.816249            75.096233           1.756868            2.106720            3.231285            4.056147
1 2026-04-27 75.002998           76.231000            78.033199            76.089149            74.899200           1.108207            2.182061            2.885774            3.904530
2 2026-04-28 73.205002           75.589799            77.414600            76.233200            74.663933           1.730684            2.592590            2.644682            3.780871
3 2026-04-29 71.569000           74.325000            76.622400            76.077150            74.398566           1.926433            3.056460            2.826368            3.706202
4 2026-04-30 73.533997           7

## 5. Feature Engineering: RSI(14)

RSI (Relative Strength Index) mengukur momentum harga.

**Formula:**
```
delta[t]   = silver[t] - silver[t-1]
avg_up     = mean(max(delta, 0)) selama 14 periode
avg_down   = mean(max(-delta, 0)) selama 14 periode
RS         = avg_up / avg_down
RSI        = 100 - (100 / (1 + RS))
```

RSI > 70 → overbought, RSI < 30 → oversold.
Fitur di-**shift(1)** sehingga hanya menggunakan informasi t-1 ke belakang.

In [6]:
rsi_df = df_demo[['ds','silver','silver_rsi_14']].copy()
# Also compute raw (unshifted) for reference
delta_s = df_all['silver'].diff()
up_s    = delta_s.clip(lower=0)
dn_s    = (-delta_s).clip(lower=0)
rsi_raw_full = (100 - 100 / (1 + up_s.rolling(14).mean() /
                              dn_s.rolling(14).mean().replace(0,1e-9)))

print('RSI(14) - 10 baris demo:')
print('  silver_rsi_14 = shift(1) dari RSI, artinya RSI yang dihitung di hari sebelumnya')
print(rsi_df.to_string())
print()

# Step-by-step for last row (t), before shift
last = len(df_all) - 1
win15 = df_all['silver'].iloc[last-14:last+1].values
deltas14 = np.diff(win15)
ups14    = np.where(deltas14 > 0, deltas14, 0.0)
dns14    = np.where(deltas14 < 0, -deltas14, 0.0)
avg_up_m  = np.mean(ups14)
avg_dn_m  = np.mean(dns14) if np.mean(dns14) > 0 else 1e-9
rs_m      = avg_up_m / avg_dn_m
rsi_m     = 100 - 100 / (1 + rs_m)

print('Perhitungan manual RSI untuk baris terakhir (t, sebelum shift):')
print(f'  15 harga silver: {[round(x,4) for x in win15]}')
print(f'  14 delta: {[round(x,6) for x in deltas14]}')
print(f'  Up changes:   {[round(x,6) for x in ups14]}')
print(f'  Down changes: {[round(x,6) for x in dns14]}')
print(f'  avg_up   = {avg_up_m:.8f}')
print(f'  avg_down = {avg_dn_m:.8f}')
print(f'  RS  = {avg_up_m:.8f} / {avg_dn_m:.8f} = {rs_m:.8f}')
print(f'  RSI = 100 - 100/(1+{rs_m:.8f}) = {rsi_m:.8f}')
print(f'  (nilai ini menjadi silver_rsi_14 untuk baris BERIKUTNYA karena shift(1))')

RSI(14) - 10 baris demo:
  silver_rsi_14 = shift(1) dari RSI, artinya RSI yang dihitung di hari sebelumnya
          ds    silver  silver_rsi_14
0 2026-04-24 76.383003      55.825861
1 2026-04-27 75.002998      57.666644
2 2026-04-28 73.205002      56.400346
3 2026-04-29 71.569000      45.652270
4 2026-04-30 73.533997      40.110066
5 2026-05-01 75.950996      44.576199
6 2026-05-04 73.071999      50.782840
7 2026-05-05 73.108002      38.008115
8 2026-05-06 76.810997      37.857174
9 2026-05-07 80.970001      46.915903

Perhitungan manual RSI untuk baris terakhir (t, sebelum shift):
  15 harga silver: [np.float64(81.738), np.float64(79.951), np.float64(76.411), np.float64(77.893), np.float64(75.465), np.float64(76.383), np.float64(75.003), np.float64(73.205), np.float64(71.569), np.float64(73.534), np.float64(75.951), np.float64(73.072), np.float64(73.108), np.float64(76.811), np.float64(80.97)]
  14 delta: [np.float64(-1.787003), np.float64(-3.539993), np.float64(1.481995), np.float64

## 6. Feature Engineering: MACD

MACD (Moving Average Convergence Divergence) mengukur momentum berbasis EMA.

**Formula:**
```
alpha_12 = 2 / (12 + 1) = 0.1538
alpha_26 = 2 / (26 + 1) = 0.0741
EMA_12[t] = alpha_12 * price[t] + (1 - alpha_12) * EMA_12[t-1]
EMA_26[t] = alpha_26 * price[t] + (1 - alpha_26) * EMA_26[t-1]
MACD[t]   = EMA_12[t] - EMA_26[t]
Signal[t] = EMA_9(MACD)
```

Semua di-**shift(1)** untuk mencegah look-ahead.

In [7]:
ema12_full = df_all['silver'].ewm(span=12, adjust=False).mean()
ema26_full = df_all['silver'].ewm(span=26, adjust=False).mean()
macd_full  = ema12_full - ema26_full

alpha12 = 2.0 / (12 + 1)
alpha26 = 2.0 / (26 + 1)
alpha9  = 2.0 / (9 + 1)
print(f'alpha_12 = 2/(12+1) = {alpha12:.6f}')
print(f'alpha_26 = 2/(26+1) = {alpha26:.6f}')
print(f'alpha_9  = 2/(9+1)  = {alpha9:.6f}')
print()

macd_display = pd.DataFrame({
    'ds':                df_demo['ds'].values,
    'silver':            df_demo['silver'].values,
    'ema12':             ema12_full.tail(N_SAMPLE).values,
    'ema26':             ema26_full.tail(N_SAMPLE).values,
    'macd_raw':          macd_full.tail(N_SAMPLE).values,
    'silver_macd':       df_demo['silver_macd'].values,
    'silver_macd_signal':df_demo['silver_macd_signal'].values,
})
print('MACD - 10 baris demo:')
print(macd_display.to_string())
print()

# Manual for last 3 rows
last = len(df_all) - 1
print('Manual EMA untuk 3 baris terakhir:')
for i in [last-2, last-1, last]:
    sv   = df_all['silver'].iloc[i]
    e12  = ema12_full.iloc[i]
    e12p = ema12_full.iloc[i-1]
    e26  = ema26_full.iloc[i]
    e26p = ema26_full.iloc[i-1]
    print(f'  t={i}: price={sv:.4f}')
    print(f'    EMA12 = {alpha12:.4f}x{sv:.4f} + {1-alpha12:.4f}x{e12p:.4f} = {e12:.6f}')
    print(f'    EMA26 = {alpha26:.4f}x{sv:.4f} + {1-alpha26:.4f}x{e26p:.4f} = {e26:.6f}')
    print(f'    MACD  = {e12:.6f} - {e26:.6f} = {macd_full.iloc[i]:.6f}')

alpha_12 = 2/(12+1) = 0.153846
alpha_26 = 2/(26+1) = 0.074074
alpha_9  = 2/(9+1)  = 0.200000

MACD - 10 baris demo:
          ds    silver     ema12     ema26  macd_raw  silver_macd  silver_macd_signal
0 2026-04-24 76.383003 77.042423 76.979513  0.062909     0.135083           -0.274950
1 2026-04-27 75.002998 76.728665 76.833105 -0.104440     0.062909           -0.207378
2 2026-04-28 73.205002 76.186563 76.564356 -0.377793    -0.104440           -0.186790
3 2026-04-29 71.569000 75.476169 76.194330 -0.718161    -0.377793           -0.224991
4 2026-04-30 73.533997 75.177373 75.997268 -0.819895    -0.718161           -0.323625
5 2026-05-01 75.950996 75.296392 75.993841 -0.697449    -0.819895           -0.422879
6 2026-05-04 73.071999 74.954178 75.777408 -0.823230    -0.697449           -0.477793
7 2026-05-05 73.108002 74.670151 75.579674 -0.909524    -0.823230           -0.546880
8 2026-05-06 76.810997 74.999512 75.670883 -0.671372    -0.909524           -0.619409
9 2026-05-07 80.970001 7

## 7. Feature Engineering: Bollinger Band Position & ATR(14)

**Bollinger Band Position** — posisi harga relatif terhadap pita Bollinger 20-hari:
```
BB_pos[t] = (price[t] - mean_20[t]) / std_20[t]
```
BB_pos = 0 → di tengah, BB_pos = +2 → upper band, BB_pos = -2 → lower band.

**ATR(14) — Average True Range**:
```
ATR_14[t] = mean(|price[t] - price[t-1]|) untuk 14 periode terakhir
```

Keduanya di-**shift(1)**.

In [8]:
bb_atr_df = pd.DataFrame({
    'ds':            df_demo['ds'].values,
    'silver':        df_demo['silver'].values,
    'silver_bb_pos': df_demo['silver_bb_pos'].values,
    'silver_atr_14': df_demo['silver_atr_14'].values,
})
print('Bollinger Band Position & ATR(14) - 10 baris demo:')
print(bb_atr_df.to_string())
print()

# Manual for last row (t), before shift
last = len(df_all) - 1
w20  = df_all['silver'].iloc[last-19:last+1].values
mean20 = np.mean(w20)
std20  = np.std(w20, ddof=1)
bb_m   = (df_all['silver'].iloc[last] - mean20) / (std20 if std20 > 0 else 1e-9)
w14d   = np.abs(np.diff(df_all['silver'].iloc[last-14:last+1].values))
atr_m  = np.mean(w14d)

print(f'Manual untuk baris terakhir (t={last}, sebelum shift):')
print(f'  20 harga silver: {[round(x,4) for x in w20[:5]]} ... {[round(x,4) for x in w20[-5:]]}')
print(f'  mean_20 = {mean20:.6f}')
print(f'  std_20  = {std20:.6f}')
print(f'  BB_pos  = ({df_all["silver"].iloc[last]:.6f} - {mean20:.6f}) / {std20:.6f} = {bb_m:.6f}')
print(f'  14 |delta|: {[round(x,6) for x in w14d]}')
print(f'  ATR_14 = mean = {atr_m:.6f}')

Bollinger Band Position & ATR(14) - 10 baris demo:
          ds    silver  silver_bb_pos  silver_atr_14
0 2026-04-24 76.383003       0.022775       1.673570
1 2026-04-27 75.002998       0.175396       1.733856
2 2026-04-28 73.205002      -0.376381       1.772785
3 2026-04-29 71.569000      -1.145014       1.658498
4 2026-04-30 73.533997      -1.595033       1.700141
5 2026-05-01 75.950996      -0.841638       1.837141
6 2026-05-04 73.071999      -0.061225       1.952570
7 2026-05-05 73.108002      -1.113840       1.881927
8 2026-05-06 76.810997      -1.162981       1.877356
9 2026-05-07 80.970001       0.197881       2.078642

Manual untuk baris terakhir (t=123, sebelum shift):
  20 harga silver: [np.float64(76.324), np.float64(75.523), np.float64(79.391), np.float64(79.491), np.float64(78.606)] ... [np.float64(75.951), np.float64(73.072), np.float64(73.108), np.float64(76.811), np.float64(80.97)]
  mean_20 = 76.519949
  std_20  = 2.855667
  BB_pos  = (80.970001 - 76.519949) / 2.855667

## 8. Feature Engineering: Cross-Asset Ratios

Rasio antar aset menangkap hubungan relatif:

| Fitur | Formula | Makna |
|-------|---------|-------|
| `gold_silver_ratio_lag_1` | gold[t-1] / silver[t-1] | Gold/Silver Ratio kemarin |
| `gold_silver_ratio_lag_5` | gold[t-5] / silver[t-5] | G/S Ratio 1 minggu lalu |
| `silver_oil_ratio_lag_1` | silver[t-1] / oil[t-1] | Silver/Oil Ratio kemarin |

Semua sudah di-lag (hanya data masa lalu).

In [9]:
ratio_df = pd.DataFrame({
    'ds':                        df_demo['ds'].values,
    'silver':                    df_demo['silver'].values,
    'gold':                      df_demo['gold'].values,
    'oil':                       df_demo['oil'].values,
    'gold_silver_ratio_lag_1':   df_demo['gold_silver_ratio_lag_1'].values,
    'gold_silver_ratio_lag_5':   df_demo['gold_silver_ratio_lag_5'].values,
    'silver_oil_ratio_lag_1':    df_demo['silver_oil_ratio_lag_1'].values,
})
print('Cross-Asset Ratio Features - 10 baris demo:')
print(ratio_df.to_string())
print()

# Manual for last row
last = len(df_clean) - 1
g1  = df_clean['gold'].iloc[last-1]
s1  = df_clean['silver'].iloc[last-1]
s5  = df_clean['silver'].iloc[last-5]
g5  = df_clean['gold'].iloc[last-5]
o1  = df_clean['oil'].iloc[last-1]
print(f'Verifikasi baris terakhir:')
print(f'  gold_silver_ratio_lag_1 = gold[t-1]/silver[t-1] = {g1:.4f}/{s1:.4f} = {g1/s1:.6f}')
print(f'  gold_silver_ratio_lag_5 = gold[t-5]/silver[t-5] = {g5:.4f}/{s5:.4f} = {g5/s5:.6f}')
print(f'  silver_oil_ratio_lag_1  = silver[t-1]/oil[t-1] = {s1:.4f}/{o1:.4f} = {s1/o1:.6f}')

Cross-Asset Ratio Features - 10 baris demo:
          ds    silver        gold        oil  gold_silver_ratio_lag_1  gold_silver_ratio_lag_5  silver_oil_ratio_lag_1
0 2026-04-24 76.383003 4722.299805  94.400002                62.348113                59.428909                0.787324
1 2026-04-27 75.002998 4675.399902  96.370003                61.823961                60.119327                0.809142
2 2026-04-28 73.205002 4591.500000  99.930000                62.336173                61.488525                0.778282
3 2026-04-29 71.569000 4545.200195 106.879997                62.721124                60.756424                0.732563
4 2026-04-30 73.533997 4614.700195 105.070000                63.507946                62.348113                0.669620
5 2026-05-01 75.950996 4629.899902 101.940002                62.756010                61.823961                0.699857
6 2026-05-04 73.071999 4519.500000 106.419998                60.959041                62.336173                0.745

## 9. Feature Engineering: Calendar Features

- **`day_of_week`**: 0=Senin, 1=Selasa, ..., 4=Jumat
- **`month`**: 1–12

Fitur kalender membantu model menangkap efek hari-dalam-minggu dan musiman bulanan.

In [10]:
cal_df = pd.DataFrame({
    'ds':          df_demo['ds'].values,
    'silver':      df_demo['silver'].values,
    'day_of_week': df_demo['day_of_week'].values.astype(int),
    'day_name':    [d.strftime('%A') for d in pd.to_datetime(df_demo['ds'])],
    'month':       df_demo['month'].values.astype(int),
    'month_name':  [d.strftime('%B') for d in pd.to_datetime(df_demo['ds'])],
})
print('Calendar Features - 10 baris demo:')
print(cal_df.to_string())

Calendar Features - 10 baris demo:
          ds    silver  day_of_week   day_name  month month_name
0 2026-04-24 76.383003            4     Friday      4      April
1 2026-04-27 75.002998            0     Monday      4      April
2 2026-04-28 73.205002            1    Tuesday      4      April
3 2026-04-29 71.569000            2  Wednesday      4      April
4 2026-04-30 73.533997            3   Thursday      4      April
5 2026-05-01 75.950996            4     Friday      5        May
6 2026-05-04 73.071999            0     Monday      5        May
7 2026-05-05 73.108002            1    Tuesday      5        May
8 2026-05-06 76.810997            2  Wednesday      5        May
9 2026-05-07 80.970001            3   Thursday      5        May


## 10. Prophet: Fitting & Dekomposisi Komponen

Prophet mendekomposisi deret waktu:
```
y(t) = trend(t) * (1 + weekly(t)) * (1 + yearly(t))   [multiplicative mode]
```

**Konfigurasi:**
- `seasonality_mode = 'multiplicative'`
- `changepoint_prior_scale = 0.15`
- `weekly_seasonality = True, yearly_seasonality = True`

**Penting:** Prophet dilatih pada **8 baris train** untuk demo ini.
Dengan data sebanyak ini, yearly seasonality tidak dapat dipelajari dengan baik — ini normal.
Dalam produksi, Prophet dilatih pada ~2500 baris (10 tahun data).

Output Prophet yang digunakan sebagai fitur: `ph_yhat`, `ph_trend`, `ph_weekly`, `ph_yearly`

In [11]:
# Train/test split
df_train = df_demo.iloc[:N_TRAIN].copy().reset_index(drop=True)
df_test  = df_demo.iloc[N_TRAIN:].copy().reset_index(drop=True)

print(f'Train: {N_TRAIN} baris ({df_train["ds"].iloc[0].date()} s/d {df_train["ds"].iloc[-1].date()})')
print(f'Test : {N_TEST} baris  ({df_test["ds"].iloc[0].date()} s/d {df_test["ds"].iloc[-1].date()})')
print()

prophet_input = df_train[['ds','silver']].rename(columns={'silver':'y'})
print('Data input ke Prophet (y = harga silver):')
print(prophet_input.to_string())
print()

m_prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.15,
    seasonality_mode='multiplicative',
    uncertainty_samples=0,
)
m_prophet.fit(prophet_input)
print('Prophet berhasil di-fit.')

Train: 8 baris (2026-04-24 s/d 2026-05-05)
Test : 2 baris  (2026-05-06 s/d 2026-05-07)

Data input ke Prophet (y = harga silver):
          ds         y
0 2026-04-24 76.383003
1 2026-04-27 75.002998
2 2026-04-28 73.205002
3 2026-04-29 71.569000
4 2026-04-30 73.533997
5 2026-05-01 75.950996
6 2026-05-04 73.071999
7 2026-05-05 73.108002



18:04:43 - cmdstanpy - INFO - Chain [1] start processing
18:04:51 - cmdstanpy - INFO - Chain [1] done processing


Prophet berhasil di-fit.


In [12]:
# Predict for all 10 demo rows
ph_fc = m_prophet.predict(df_demo[['ds']].copy())
ph_comp = ph_fc[['ds','yhat','trend']].copy()
ph_comp['weekly'] = ph_fc['weekly'] if 'weekly' in ph_fc.columns else 0.0
ph_comp['yearly'] = ph_fc['yearly'] if 'yearly' in ph_fc.columns else 0.0
ph_comp.insert(1, 'silver_actual', df_demo['silver'].values)
ph_comp.columns = ['ds','silver_actual','ph_yhat','ph_trend','ph_weekly','ph_yearly']

print('Komponen Prophet untuk 10 baris demo:')
print(ph_comp.to_string())
print()
print('Catatan kolom:')
print('  ph_yhat   : prediksi Prophet')
print('  ph_trend  : komponen tren jangka panjang')
print('  ph_weekly : efek hari-dalam-minggu')
print('  ph_yearly : efek musiman tahunan')

# Add Prophet features to df_demo
df_demo['ph_yhat']   = ph_fc['yhat'].values
df_demo['ph_trend']  = ph_fc['trend'].values
df_demo['ph_weekly'] = ph_fc['weekly'].values if 'weekly' in ph_fc.columns else 0.0
df_demo['ph_yearly'] = ph_fc['yearly'].values if 'yearly' in ph_fc.columns else 0.0
print('Fitur Prophet ditambahkan ke df_demo.')

Komponen Prophet untuk 10 baris demo:
          ds  silver_actual   ph_yhat  ph_trend  ph_weekly  ph_yearly
0 2026-04-24      76.383003 76.383000 58.082187  -1.163704   1.478789
1 2026-04-27      75.002998 75.002996 59.596890  -1.221119   1.479624
2 2026-04-28      73.205002 73.205003 60.101808  -1.243295   1.461311
3 2026-04-29      71.569000 71.569006 60.606710  -1.258215   1.439091
4 2026-04-30      73.533997 73.534002 61.111556  -1.213084   1.416359
5 2026-05-01      75.950996 75.951000 61.616429  -1.163704   1.396346
6 2026-05-04      73.071999 73.072002 63.130986  -1.221119   1.378585
7 2026-05-05      73.108002 73.108008 63.635839  -1.243295   1.392144
8 2026-05-06      76.810997 74.263444 64.140692  -1.258215   1.416036
9 2026-05-07      80.970001 79.907252 64.645544  -1.213084   1.449167

Catatan kolom:
  ph_yhat   : prediksi Prophet
  ph_trend  : komponen tren jangka panjang
  ph_weekly : efek hari-dalam-minggu
  ph_yearly : efek musiman tahunan
Fitur Prophet ditambahkan ke d

## 11. Target: Log Return 1-Step

Model tidak memprediksi harga langsung, melainkan **log return**:
```
y[t] = log(silver[t] / silver[t-1])
```

**Mengapa log return?**
- Lebih stasioner dari harga absolut
- Simetris: +10% dan -10% memiliki magnitude sama
- Mudah dibalik: `price[t] = price[t-1] × exp(y_pred[t])`

In [13]:
silver_t  = df_demo['silver'].values
silver_t1 = df_demo['silver_lag_1'].values  # silver[t-1] dari fitur lag

log_ret = np.where(silver_t1 > 0, np.log(silver_t / silver_t1), 0.0)

logret_df = pd.DataFrame({
    'ds':          df_demo['ds'].values,
    'silver[t]':   silver_t,
    'silver[t-1]': silver_t1,
    'ratio':       silver_t / silver_t1,
    'log_return':  log_ret,
    'split':       ['TRAIN'] * N_TRAIN + ['TEST'] * N_TEST,
})
print('Log Return Target:')
print(logret_df.to_string())
print()

# Manual verification
print('Verifikasi manual (rumus: log(silver[t] / silver[t-1])):')
for i in [0, 1, 7]:
    st  = silver_t[i]
    st1 = silver_t1[i]
    lr  = math.log(st / st1)
    print(f'  Baris {i}: log({st:.6f} / {st1:.6f}) = log({st/st1:.8f}) = {lr:.8f}')

y_train = log_ret[:N_TRAIN]
actual_test_prices = silver_t[N_TRAIN:]
prev_prices_test   = silver_t1[N_TRAIN:]

print()
print(f'y_train (8 log returns): {[round(x,8) for x in y_train]}')
print(f'Harga aktual test:       {[round(x,4) for x in actual_test_prices]}')
print(f'Harga sebelumnya (t-1):  {[round(x,4) for x in prev_prices_test]}')

Log Return Target:
          ds  silver[t]  silver[t-1]    ratio  log_return  split
0 2026-04-24  76.383003    75.464996 1.012165    0.012091  TRAIN
1 2026-04-27  75.002998    76.383003 0.981933   -0.018232  TRAIN
2 2026-04-28  73.205002    75.002998 0.976028   -0.024264  TRAIN
3 2026-04-29  71.569000    73.205002 0.977652   -0.022602  TRAIN
4 2026-04-30  73.533997    71.569000 1.027456    0.027086  TRAIN
5 2026-05-01  75.950996    73.533997 1.032869    0.032341  TRAIN
6 2026-05-04  73.071999    75.950996 0.962094   -0.038643  TRAIN
7 2026-05-05  73.108002    73.071999 1.000493    0.000493  TRAIN
8 2026-05-06  76.810997    73.108002 1.050651    0.049410   TEST
9 2026-05-07  80.970001    76.810997 1.054146    0.052731   TEST

Verifikasi manual (rumus: log(silver[t] / silver[t-1])):
  Baris 0: log(76.383003 / 75.464996) = log(1.01216467) = 0.01209128
  Baris 1: log(75.002998 / 76.383003) = log(0.98193309) = -0.01823211
  Baris 7: log(73.108002 / 73.071999) = log(1.00049271) = 0.00049259


## 12. Matriks Fitur Final (X)

Gabungkan semua fitur sesuai urutan di `forecaster.py`:

| Grup | Jumlah Fitur |
|------|--------------|
| Price lag (4 var × 7 lag) | 28 |
| Return lag (4 var × 7 lag) | 28 |
| Rolling mean/std (4 var × 4 window × 2) | 32 |
| Calendar (day_of_week, month) | 2 |
| Prophet (yhat, trend, weekly, yearly) | 4 |
| Technical (RSI, MACD, signal, BB, ATR) | 5 |
| Cross-asset ratios | 3 |
| **Total** | **102** |

In [14]:
price_lag_cols = [f'{s}_lag_{d}'     for s in SERIES_LIST for d in LAG_DAYS]
ret_lag_cols   = [f'{s}_ret_lag_{d}' for s in SERIES_LIST for d in LAG_DAYS]
roll_cols_all  = [f'{s}_{fn}_{w}'
                  for s in SERIES_LIST
                  for fn in ['roll_mean','roll_std']
                  for w in ROLLING_WINDOWS]
calendar_cols  = ['day_of_week', 'month']
prophet_cols   = ['ph_yhat', 'ph_trend', 'ph_weekly', 'ph_yearly']
tech_cols      = ['silver_rsi_14','silver_macd','silver_macd_signal','silver_bb_pos','silver_atr_14']
ratio_cols     = ['gold_silver_ratio_lag_1','gold_silver_ratio_lag_5','silver_oil_ratio_lag_1']
feature_cols   = price_lag_cols + ret_lag_cols + roll_cols_all + calendar_cols + prophet_cols + tech_cols + ratio_cols

print(f'Total fitur: {len(feature_cols)}')
print()

X_all   = df_demo[feature_cols].values
X_train = X_all[:N_TRAIN]
X_test  = X_all[N_TRAIN:]

print(f'X_train shape: {X_train.shape}  (8 baris × 102 fitur)')
print(f'X_test  shape: {X_test.shape}  (2 baris × 102 fitur)')
print()

# Show first 10 features of X_train as DataFrame
X_train_show = pd.DataFrame(X_train[:, :10], columns=feature_cols[:10])
X_train_show.insert(0, 'ds', df_demo['ds'].iloc[:N_TRAIN].values)
print('X_train — 10 fitur pertama (dari 102):')
print(X_train_show.to_string())
print()
# Show last 10 features
X_train_show2 = pd.DataFrame(X_train[:, -10:], columns=feature_cols[-10:])
X_train_show2.insert(0, 'ds', df_demo['ds'].iloc[:N_TRAIN].values)
print('X_train — 10 fitur terakhir (dari 102):')
print(X_train_show2.to_string())

Total fitur: 102

X_train shape: (8, 102)  (8 baris × 102 fitur)
X_test  shape: (2, 102)  (2 baris × 102 fitur)

X_train — 10 fitur pertama (dari 102):
          ds  silver_lag_1  silver_lag_2  silver_lag_3  silver_lag_5  silver_lag_7  silver_lag_10  silver_lag_14  gold_lag_1  gold_lag_2  gold_lag_3
0 2026-04-24     75.464996     77.892998     76.411003     81.737999     79.490997      76.323997      72.661003 4705.100098 4732.500000 4698.399902
1 2026-04-27     76.383003     75.464996     77.892998     79.950996     78.606003      75.523003      71.825996 4722.299805 4705.100098 4732.500000
2 2026-04-28     75.002998     76.383003     75.464996     76.411003     81.737999      79.390999      75.223999 4675.399902 4722.299805 4705.100098
3 2026-04-29     73.205002     75.002998     76.383003     77.892998     79.950996      79.490997      76.277000 4591.500000 4675.399902 4722.299805
4 2026-04-30     71.569000     73.205002     75.002998     75.464996     76.411003      78.606003      

## 13. Training & Prediksi XGBoost

XGBoost membangun ensemble pohon keputusan secara sekuensial (gradient boosting).

**Konfigurasi (dari `forecaster.py`):**
```python
XGBRegressor(
    n_estimators=1200, learning_rate=0.01, max_depth=6,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3
)
```

**Input X_train** (8×102), **Target y_train** (8 log returns)

**Prediksi → Log Return → Harga:**
```
log_ret_pred = xgb.predict(X_test_row)
price_pred   = price[t-1] × exp(log_ret_pred)
```

In [15]:
xgb_model = XGBRegressor(
    n_estimators=1200, learning_rate=0.01, max_depth=6,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3,
    random_state=42, verbosity=0,
)
print('Training XGBoost...')
xgb_model.fit(X_train, y_train)
print('Selesai.')
print()

xgb_log_pred   = xgb_model.predict(X_test)
xgb_price_pred = prev_prices_test * np.exp(xgb_log_pred)

print('Prediksi XGBoost untuk 2 baris test:')
for i in range(N_TEST):
    pv = prev_prices_test[i]
    lr = xgb_log_pred[i]
    pp = xgb_price_pred[i]
    at = actual_test_prices[i]
    print(f'  Baris {i}: prev_price={pv:.4f}, log_ret_pred={lr:.8f}')
    print(f'           price_pred = {pv:.4f} x exp({lr:.8f})')
    print(f'                      = {pv:.4f} x {np.exp(lr):.8f} = {pp:.4f}')
    print(f'           actual     = {at:.4f}')
    print(f'           error      = {at - pp:+.4f}')
    print()

Training XGBoost...
Selesai.

Prediksi XGBoost untuk 2 baris test:
  Baris 0: prev_price=73.1080, log_ret_pred=-0.00396639
           price_pred = 73.1080 x exp(-0.00396639)
                      = 73.1080 x 0.99604148 = 72.8186
           actual     = 76.8110
           error      = +3.9924

  Baris 1: prev_price=76.8110, log_ret_pred=-0.00396639
           price_pred = 76.8110 x exp(-0.00396639)
                      = 76.8110 x 0.99604148 = 76.5069
           actual     = 80.9700
           error      = +4.4631



## 14. Training & Prediksi LightGBM

LightGBM menggunakan histogram-based learning, lebih cepat dari XGBoost pada dataset besar.

**Konfigurasi (dari `forecaster.py`):**
```python
LGBMRegressor(
    n_estimators=2000, learning_rate=0.005, max_depth=6, num_leaves=63,
    subsample=0.8, subsample_freq=5, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=2.0, min_child_samples=15
)
```

In [16]:
lgbm_model = LGBMRegressor(
    n_estimators=2000, learning_rate=0.005, max_depth=6, num_leaves=63,
    subsample=0.8, subsample_freq=5, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=2.0, min_child_samples=15,
    random_state=42, verbose=-1,
)
print('Training LightGBM...')
lgbm_model.fit(X_train, y_train)
print('Selesai.')
print()

lgbm_log_pred   = lgbm_model.predict(X_test)
lgbm_price_pred = prev_prices_test * np.exp(lgbm_log_pred)

print('Prediksi LightGBM untuk 2 baris test:')
for i in range(N_TEST):
    pv = prev_prices_test[i]
    lr = lgbm_log_pred[i]
    pp = lgbm_price_pred[i]
    at = actual_test_prices[i]
    print(f'  Baris {i}: prev_price={pv:.4f}, log_ret_pred={lr:.8f}')
    print(f'           price_pred = {pv:.4f} x exp({lr:.8f})')
    print(f'                      = {pv:.4f} x {np.exp(lr):.8f} = {pp:.4f}')
    print(f'           actual     = {at:.4f}')
    print(f'           error      = {at - pp:+.4f}')
    print()

Training LightGBM...
Selesai.

Prediksi LightGBM untuk 2 baris test:
  Baris 0: prev_price=73.1080, log_ret_pred=-0.00396639
           price_pred = 73.1080 x exp(-0.00396639)
                      = 73.1080 x 0.99604147 = 72.8186
           actual     = 76.8110
           error      = +3.9924

  Baris 1: prev_price=76.8110, log_ret_pred=-0.00396639
           price_pred = 76.8110 x exp(-0.00396639)
                      = 76.8110 x 0.99604147 = 76.5069
           actual     = 80.9700
           error      = +4.4631



## 15. Evaluasi Metrik

| Metrik | Formula | Keterangan |
|--------|---------|------------|
| **RMSE** | √(Σ(actual−pred)² / n) | Root Mean Square Error (USD) |
| **MAE** | Σ|actual−pred| / n | Mean Absolute Error (USD) |
| **MAPE** | Σ(|actual−pred|/actual) × 100 / n | Error dalam persen (%) |
| **R²** | 1 − SS_res/SS_tot | Koefisien determinasi (1.0 = sempurna) |

Evaluasi dilakukan pada **2 baris test set**.

In [17]:
actual   = actual_test_prices
n        = len(actual)

print('=' * 65)
print('PERBANDINGAN PREDIKSI VS AKTUAL')
print('=' * 65)
comp = pd.DataFrame({
    'Tanggal':    [str(d.date()) for d in df_test['ds']],
    'Aktual':     actual,
    'XGBoost':    xgb_price_pred,
    'LightGBM':   lgbm_price_pred,
    'Err_XGB':    actual - xgb_price_pred,
    'Err_LGBM':   actual - lgbm_price_pred,
})
print(comp.to_string(index=False))

# XGBoost metrics step-by-step
print()
print('=' * 65)
print('METRIK XGBOOST — LANGKAH DEMI LANGKAH')
print('=' * 65)
xgb_sq   = (actual - xgb_price_pred) ** 2
xgb_abs  = np.abs(actual - xgb_price_pred)
xgb_pct  = xgb_abs / actual * 100

print(f'n = {n}')
print()
print('Squared Errors:')
for i in range(n):
    print(f'  ({actual[i]:.4f} - {xgb_price_pred[i]:.4f})^2 = {actual[i]-xgb_price_pred[i]:+.6f}^2 = {xgb_sq[i]:.8f}')
mse_xgb  = np.mean(xgb_sq)
rmse_xgb = np.sqrt(mse_xgb)
print(f'  MSE  = ({" + ".join([f"{x:.8f}" for x in xgb_sq])}) / {n} = {mse_xgb:.8f}')
print(f'  RMSE = sqrt({mse_xgb:.8f}) = {rmse_xgb:.8f}')
print()
print('Absolute Errors:')
for i in range(n):
    print(f'  |{actual[i]:.4f} - {xgb_price_pred[i]:.4f}| = {xgb_abs[i]:.8f}')
mae_xgb = np.mean(xgb_abs)
print(f'  MAE = ({" + ".join([f"{x:.8f}" for x in xgb_abs])}) / {n} = {mae_xgb:.8f}')
print()
print('Percentage Errors:')
for i in range(n):
    print(f'  |{actual[i]:.4f} - {xgb_price_pred[i]:.4f}| / {actual[i]:.4f} x100 = {xgb_pct[i]:.6f}%')
mape_xgb = np.mean(xgb_pct)
print(f'  MAPE = {mape_xgb:.6f}%')
print()
mean_actual = np.mean(actual)
ss_res_xgb = np.sum((actual - xgb_price_pred)**2)
ss_tot     = np.sum((actual - mean_actual)**2)
r2_xgb = (1 - ss_res_xgb / ss_tot) if ss_tot > 0 else float('nan')
print(f'R2 = 1 - SS_res / SS_tot')
print(f'   SS_res = {ss_res_xgb:.8f}')
print(f'   SS_tot = sum((actual - mean_actual)^2), mean_actual = {mean_actual:.6f}')
for i in range(n):
    print(f'     ({actual[i]:.4f} - {mean_actual:.4f})^2 = {(actual[i]-mean_actual)**2:.8f}')
print(f'   SS_tot = {ss_tot:.8f}')
print(f'   R2  = 1 - {ss_res_xgb:.8f} / {ss_tot:.8f} = {r2_xgb:.8f}')

# LightGBM metrics
print()
print('=' * 65)
print('METRIK LIGHTGBM — LANGKAH DEMI LANGKAH')
print('=' * 65)
lgbm_sq  = (actual - lgbm_price_pred) ** 2
lgbm_abs = np.abs(actual - lgbm_price_pred)
lgbm_pct = lgbm_abs / actual * 100
mse_lgbm  = np.mean(lgbm_sq)
rmse_lgbm = np.sqrt(mse_lgbm)
mae_lgbm  = np.mean(lgbm_abs)
mape_lgbm = np.mean(lgbm_pct)
ss_res_lgbm = np.sum((actual - lgbm_price_pred)**2)
r2_lgbm = (1 - ss_res_lgbm / ss_tot) if ss_tot > 0 else float('nan')

print(f'n = {n}')
print('Squared Errors:')
for i in range(n):
    print(f'  ({actual[i]:.4f} - {lgbm_price_pred[i]:.4f})^2 = {actual[i]-lgbm_price_pred[i]:+.6f}^2 = {lgbm_sq[i]:.8f}')
print(f'  RMSE = sqrt({mse_lgbm:.8f}) = {rmse_lgbm:.8f}')
print(f'  MAE  = {mae_lgbm:.8f}')
print(f'  MAPE = {mape_lgbm:.6f}%')
print(f'  R2   = 1 - {ss_res_lgbm:.8f} / {ss_tot:.8f} = {r2_lgbm:.8f}')

PERBANDINGAN PREDIKSI VS AKTUAL
   Tanggal    Aktual   XGBoost  LightGBM  Err_XGB  Err_LGBM
2026-05-06 76.810997 72.818602 72.818601 3.992395  3.992396
2026-05-07 80.970001 76.506939 76.506938 4.463062  4.463063

METRIK XGBOOST — LANGKAH DEMI LANGKAH
n = 2

Squared Errors:
  (76.8110 - 72.8186)^2 = +3.992395^2 = 15.93921805
  (80.9700 - 76.5069)^2 = +4.463062^2 = 19.91892537
  MSE  = (15.93921805 + 19.91892537) / 2 = 17.92907171
  RMSE = sqrt(17.92907171) = 4.23427346

Absolute Errors:
  |76.8110 - 72.8186| = 3.99239503
  |80.9700 - 76.5069| = 4.46306233
  MAE = (3.99239503 + 4.46306233) / 2 = 4.22772868

Percentage Errors:
  |76.8110 - 72.8186| / 76.8110 x100 = 5.197687%
  |80.9700 - 76.5069| / 80.9700 x100 = 5.511995%
  MAPE = 5.354841%

R2 = 1 - SS_res / SS_tot
   SS_res = 35.85814342
   SS_tot = sum((actual - mean_actual)^2), mean_actual = 78.890499
     (76.8110 - 78.8905)^2 = 4.32432901
     (80.9700 - 78.8905)^2 = 4.32432901
   SS_tot = 8.64865802
   R2  = 1 - 35.85814342 / 8.64

## 16. Ringkasan Akhir

Perbandingan lengkap hasil prediksi dan metrik evaluasi.

> **Catatan penting:** Metrik ini berdasarkan **2 data test saja**.
> Dalam aplikasi produksi (`forecaster.py`), model dilatih dengan ~2500 baris (10 tahun)
> dan dievaluasi pada 30+ baris test — hasil metrik jauh lebih representatif.

In [18]:
print('=' * 65)
print('RINGKASAN AKHIR')
print('=' * 65)
print()
print('Harga Prediksi vs Aktual (USD/troy oz):')
summary = pd.DataFrame({
    'Tanggal':  [str(d.date()) for d in df_test['ds']],
    'Aktual':   [f'${x:.4f}' for x in actual],
    'XGBoost':  [f'${x:.4f}' for x in xgb_price_pred],
    'LightGBM': [f'${x:.4f}' for x in lgbm_price_pred],
    'Err_XGB%': [f'{abs(actual[i]-xgb_price_pred[i])/actual[i]*100:.4f}%' for i in range(n)],
    'Err_LGBM%':[f'{abs(actual[i]-lgbm_price_pred[i])/actual[i]*100:.4f}%' for i in range(n)],
})
print(summary.to_string(index=False))
print()

print('Metrik Evaluasi:')
metrics = pd.DataFrame({
    'Metrik':   ['RMSE (USD)', 'MAE (USD)', 'MAPE (%)', 'R2'],
    'XGBoost':  [f'{rmse_xgb:.6f}',  f'{mae_xgb:.6f}',  f'{mape_xgb:.4f}',  f'{r2_xgb:.6f}'],
    'LightGBM': [f'{rmse_lgbm:.6f}', f'{mae_lgbm:.6f}', f'{mape_lgbm:.4f}', f'{r2_lgbm:.6f}'],
})
print(metrics.to_string(index=False))
print()
print('Pipeline selesai. Semua angka telah ditampilkan di atas.')

RINGKASAN AKHIR

Harga Prediksi vs Aktual (USD/troy oz):
   Tanggal   Aktual  XGBoost LightGBM Err_XGB% Err_LGBM%
2026-05-06 $76.8110 $72.8186 $72.8186  5.1977%   5.1977%
2026-05-07 $80.9700 $76.5069 $76.5069  5.5120%   5.5120%

Metrik Evaluasi:
    Metrik   XGBoost  LightGBM
RMSE (USD)  4.234273  4.234274
 MAE (USD)  4.227729  4.227729
  MAPE (%)    5.3548    5.3548
        R2 -3.146093 -3.146095

Pipeline selesai. Semua angka telah ditampilkan di atas.
